## UrbanRide_06 — Silver Clickstream
**Method:** Batch transformation — Bronze → Silver  
**Input:** `urbanride.bronze.clickstream`  
**Output:** `urbanride.silver.clickstream`  
**Schedule:** Daily · runs after UrbanRide_02 (Bronze Events ingest)

**Load strategy:**
- First run → full load
- Daily runs → incremental append
- Watermark: `_ingested_at` (Bronze) vs `_processed_at` (Silver)
- Target grain: one clean row per `event_id`

**Transformations:**
- Deduplication on `event_id`
- Cast `event_timestamp` STRING → TIMESTAMP
- Validate `event_type` — only known values
- Flag sessions where `app_open` is not the first event
- Flag sessions with only 1 event
- Partition by `event_type`

In [0]:
from pyspark.sql.functions import (
    col, when, lit, current_timestamp,
    to_timestamp, count, min as spark_min,
    first, row_number
)
from pyspark.sql.window import Window

CATALOG = 'urbanride'
BRONZE  = f'{CATALOG}.bronze'
SILVER  = f'{CATALOG}.silver'

spark.sql(f'USE CATALOG {CATALOG}')

VALID_EVENT_TYPES = ['app_open', 'search_ride', 'view_price', 'confirm_booking', 'cancel_booking']
VALID_AB_GROUPS   = ['control', 'variant']
VALID_DEVICES     = ['Android', 'iOS', 'Web']

print(f'Catalog : {CATALOG}')
print(f'Source  : {BRONZE}.clickstream')
print(f'Target  : {SILVER}.clickstream')
print('Config ready.')


Catalog : urbanride
Source  : urbanride.bronze.clickstream
Target  : urbanride.silver.clickstream
Config ready.


In [0]:
print('Checking load type...')

table_exists = spark.catalog.tableExists(f'{SILVER}.clickstream')

if not table_exists:
    LOAD_TYPE = 'full'
    last_run  = None
    print('  Silver table does not exist — FULL LOAD')
else:
    LOAD_TYPE = 'incremental'
    last_run  = spark.sql(
        f'SELECT MAX(_processed_at) FROM {SILVER}.clickstream'
    ).collect()[0][0]
    print(f'  Silver table exists — INCREMENTAL LOAD')
    print(f'  Last processed at : {last_run}')

print(f'  Load type : {LOAD_TYPE}')


Checking load type...
  Silver table does not exist — FULL LOAD
  Load type : full


In [0]:
print('Reading bronze.clickstream...')

df_bronze_full = spark.table(f'{BRONZE}.clickstream')

if LOAD_TYPE == 'full':
    df_bronze = df_bronze_full
    print(f'  Full load — reading all rows')
else:
    df_bronze = df_bronze_full.filter(col('_ingested_at') > last_run)
    print(f'  Incremental — reading rows ingested after {last_run}')

input_count = df_bronze.count()
print(f'  Rows to process : {input_count:,}')

if input_count == 0:
    print('  No new rows — Silver already up to date.')
    dbutils.notebook.exit('No new rows — skipping')

# Profile before cleaning
print()
print('Bronze profile (before cleaning):')
print(f'  Duplicate event_ids    : {input_count - df_bronze.dropDuplicates(["event_id"]).count():,}')
print(f'  Invalid event types    : {df_bronze.filter(~col("event_type").isin(VALID_EVENT_TYPES)).count():,}')
print(f'  Invalid ab_test_group  : {df_bronze.filter(~col("ab_test_group").isin(VALID_AB_GROUPS)).count():,}')
print()
print('Event type distribution:')
df_bronze.groupBy('event_type').count().orderBy('count', ascending=False).show()
print('A/B group distribution:')
df_bronze.groupBy('ab_test_group').count().show()
print('Device distribution:')
df_bronze.groupBy('device_type').count().orderBy('count', ascending=False).show()


Reading bronze.clickstream...
  Full load — reading all rows
  Rows to process : 24,407,534

Bronze profile (before cleaning):
  Duplicate event_ids    : 0
  Invalid event types    : 0
  Invalid ab_test_group  : 0

Event type distribution:
+---------------+--------+
|     event_type|   count|
+---------------+--------+
|       app_open|11119308|
|    search_ride| 7228714|
|     view_price| 4336375|
| cancel_booking| 1246318|
|confirm_booking|  476819|
+---------------+--------+

A/B group distribution:
+-------------+--------+
|ab_test_group|   count|
+-------------+--------+
|      control|12234323|
|      variant|12173211|
+-------------+--------+

Device distribution:
+-----------+--------+
|device_type|   count|
+-----------+--------+
|    Android|17007802|
|        iOS| 6170025|
|        Web| 1229707|
+-----------+--------+



In [0]:
print('Deduplicating on event_id...')

df_deduped = df_bronze.dropDuplicates(['event_id'])

removed = input_count - df_deduped.count()
print(f'  Rows before : {input_count:,}')
print(f'  Rows after  : {df_deduped.count():,}')
print(f'  Removed     : {removed:,} duplicate rows')


Deduplicating on event_id...
  Rows before : 24,407,534
  Rows after  : 24,407,534
  Removed     : 0 duplicate rows


In [0]:
# event_timestamp is STRING in Bronze — Auto Loader stored raw value
# Cast to TIMESTAMP here in Silver
print('Casting event_timestamp...')

df_typed = df_deduped.withColumn(
    'event_timestamp',
    to_timestamp(col('event_timestamp'), 'yyyy-MM-dd HH:mm:ss')
)

bad_ts = df_typed.filter(col('event_timestamp').isNull()).count()
print(f'  NULL timestamps after cast : {bad_ts:,}')

if bad_ts > 0:
    print('  WARNING — some timestamps could not be parsed')
    df_typed.filter(col('event_timestamp').isNull()) \
        .select('event_id','event_timestamp').show(5)


Casting event_timestamp...
  NULL timestamps after cast : 0


In [0]:
print('Validating event types...')

df_validated = df_typed.withColumn(
    'is_event_type_invalid',
    ~col('event_type').isin(VALID_EVENT_TYPES)
)

invalid_count = df_validated.filter(col('is_event_type_invalid') == True).count()
print(f'  Invalid event type rows : {invalid_count:,}')

if invalid_count > 0:
    print('  Unknown event types found:')
    df_validated.filter(col('is_event_type_invalid') == True) \
        .groupBy('event_type').count().show()


Validating event types...
  Invalid event type rows : 0


### What is a Session
A session is a single user interaction with the app from open to end. Every time a customer opens the UrbanRide app, a new session_id is generated.

Customer opens app     → session_id = "sess_001" starts   
Customer searches ride → same session_id   
Customer views price   → same session_id   
Customer books ride    → same session_id   
Customer closes app    → session ends   

Next time they open    → new session_id = "sess_002"

event_id  | session_id | customer_id | event_type      | event_timestamp
----------|------------|-------------|-----------------|--------------------
e001      | sess_001   | c001        | app_open        | 2025-09-01 08:00:00
e002      | sess_001   | c001        | search_ride     | 2025-09-01 08:00:45
e003      | sess_001   | c001        | view_price      | 2025-09-01 08:01:10
e004      | sess_001   | c001        | confirm_booking | 2025-09-01 08:01:45
e005      | sess_002   | c001        | app_open        | 2025-09-01 18:00:00
e006      | sess_002   | c001        | search_ride     | 2025-09-01 18:00:30
e007      | sess_002   | c001        | cancel_booking  | 2025-09-01 18:01:00

sess_001 — complete session, customer booked a ride.   
sess_002 — complete session, customer cancelled.

**Flag 1 — Invalid Session (first event != app_open)**   
Business rule: Every session must start with app_open. If it doesn't — something went wrong with event tracking. Either the app_open event was lost, or events are being mislabeled.

event_id  | session_id | event_type      | event_timestamp
----------|------------|-----------------|--------------------
e010      | sess_003   | search_ride     | 2025-09-01 09:00:00  ← first event
e011      | sess_003   | view_price      | 2025-09-01 09:00:30
e012      | sess_003   | confirm_booking | 2025-09-01 09:01:00

sess_003 starts with search_ride — app_open is missing. This session is flagged is_session_invalid = True on all 3 rows.

In [0]:
# Flag 1 — sessions where app_open is not the first event
# Business rule: every session must start with app_open
# Flag 2 — sessions with only 1 event (incomplete sessions)
print('Computing session quality flags...')

# Window — order events within each session by timestamp
w_session = Window.partitionBy('session_id').orderBy('event_timestamp')
w_session_count = Window.partitionBy('session_id')

df_session = df_validated \
    .withColumn('session_first_event',
        first('event_type').over(w_session)
    ) \
    .withColumn('session_event_count',
        count('event_id').over(w_session_count)
    ) \
    .withColumn('is_session_invalid',
        # Session is invalid if first event is not app_open
        col('session_first_event') != 'app_open'
    ) \
    .withColumn('is_single_event_session',
        # Sessions with only 1 event — user opened app and nothing else
        col('session_event_count') == 1
    )

invalid_sessions = df_session \
    .select('session_id','is_session_invalid') \
    .dropDuplicates(['session_id']) \
    .filter(col('is_session_invalid') == True).count()

single_sessions = df_session \
    .select('session_id','is_single_event_session') \
    .dropDuplicates(['session_id']) \
    .filter(col('is_single_event_session') == True).count()

total_sessions = df_session.select('session_id').dropDuplicates().count()

print(f'  Total sessions           : {total_sessions:,}')
print(f'  Invalid sessions         : {invalid_sessions:,}  (first event != app_open)')
print(f'  Single event sessions    : {single_sessions:,}  (incomplete sessions)')


Computing session quality flags...
  Total sessions           : 11,119,308
  Invalid sessions         : 0  (first event != app_open)
  Single event sessions    : 3,287,634  (incomplete sessions)


In [0]:
# Drop helper columns not needed in final Silver table
df_silver = df_session \
    .drop('session_first_event') \
    .withColumn('_processed_at', current_timestamp()) \
    .withColumn('_source', lit(f'{BRONZE}.clickstream'))

print(f'Final row count   : {df_silver.count():,}')
print(f'Final column count: {len(df_silver.columns)}')
print()
df_silver.printSchema()


Final row count   : 24,407,534
Final column count: 16

root
 |-- event_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- city: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- ab_test_group: string (nullable = true)
 |-- ab_test_name: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- is_event_type_invalid: boolean (nullable = true)
 |-- session_event_count: long (nullable = false)
 |-- is_session_invalid: boolean (nullable = true)
 |-- is_single_event_session: boolean (nullable = false)
 |-- _processed_at: timestamp (nullable = false)
 |-- _source: string (nullable = false)



In [0]:
import time
print(f'Writing silver.clickstream — mode: {LOAD_TYPE}...')
t0 = time.time()

if LOAD_TYPE == 'full':
    df_silver.write \
        .format('delta') \
        .mode('overwrite') \
        .partitionBy('event_type') \
        .option('overwriteSchema', 'true') \
        .saveAsTable(f'{SILVER}.clickstream')
    print(f'  Full load written : {df_silver.count():,} rows')
    print(f'  Write time : {time.time()-t0}s')
    print('Running OPTIMIZE...')
    spark.sql(f'OPTIMIZE {SILVER}.clickstream')
    print('  OPTIMIZE complete')
else:
    new_count = df_silver.count()
    if new_count > 0:
        df_silver.write \
            .format('delta') \
            .mode('append') \
            .saveAsTable(f'{SILVER}.clickstream')
        print(f'  Incremental rows appended : {new_count:,}')
        print(f'  Write time : {time.time()-t0}s')
        print('Running OPTIMIZE...')
        spark.sql(f'OPTIMIZE {SILVER}.clickstream')
        print('  OPTIMIZE complete')
    else:
        print('  No new rows — skipping write and OPTIMIZE')


Writing silver.clickstream — mode: full...
  Full load written : 24,407,534 rows
  Write time : 45.36172151565552s
Running OPTIMIZE...
  OPTIMIZE complete


In [0]:
print('=== SILVER CLICKSTREAM VERIFICATION ===')
print()

df_verify = spark.table(f'{SILVER}.clickstream')
total = df_verify.count()

print(f'  Total rows : {total:,}')
print(f'  Load type  : {LOAD_TYPE}')
print()

print('Data quality checks:')
checks = [
    ('Null event_id',          df_verify.filter(col('event_id').isNull()).count(),              False),
    ('Null event_timestamp',   df_verify.filter(col('event_timestamp').isNull()).count(),       False),
    ('Null customer_id',       df_verify.filter(col('customer_id').isNull()).count(),           False),
    ('Invalid event type',     df_verify.filter(col('is_event_type_invalid') == True).count(), False),
    ('Invalid sessions',       df_verify.filter(col('is_session_invalid') == True).count(),    True),
    ('Single event sessions',  df_verify.filter(col('is_single_event_session') == True).count(), True),
]

for check, result, is_signal in checks:
    if is_signal:
        status = 'INFO'
    else:
        status = 'WARN' if result > 0 else 'PASS'
    print(f'  {status}  {check:<30} : {result:,}')

print()
print('Event type distribution:')
df_verify.groupBy('event_type').count().orderBy('count', ascending=False).show()

print('A/B test group distribution:')
df_verify.groupBy('ab_test_group').count().show()

print('Device distribution:')
df_verify.groupBy('device_type').count().orderBy('count', ascending=False).show()

print('Sample rows:')
df_verify.select(
    'event_id','customer_id','session_id','event_type',
    'event_timestamp','ab_test_group',
    'session_event_count','is_session_invalid',
    'is_single_event_session'
).limit(5).show(truncate=False)

print()
print('Silver clickstream ready.')
print('Bronze → Silver complete. Next: Gold layer notebooks.')


=== SILVER CLICKSTREAM VERIFICATION ===

  Total rows : 24,407,534
  Load type  : full

Data quality checks:
  PASS  Null event_id                  : 0
  PASS  Null event_timestamp           : 0
  PASS  Null customer_id               : 0
  PASS  Invalid event type             : 0
  INFO  Invalid sessions               : 0
  INFO  Single event sessions          : 3,287,634

Event type distribution:
+---------------+--------+
|     event_type|   count|
+---------------+--------+
|       app_open|11119308|
|    search_ride| 7228714|
|     view_price| 4336375|
| cancel_booking| 1246318|
|confirm_booking|  476819|
+---------------+--------+

A/B test group distribution:
+-------------+--------+
|ab_test_group|   count|
+-------------+--------+
|      control|12234323|
|      variant|12173211|
+-------------+--------+

Device distribution:
+-----------+--------+
|device_type|   count|
+-----------+--------+
|    Android|17007802|
|        iOS| 6170025|
|        Web| 1229707|
+-----------+---